# M1-03 — Kaggle vLLM Node 1 acceptance

Mục tiêu notebook:

1. xác minh Kaggle T4 + stack runtime;
2. clone đúng branch `feat/m1-03-mc-ocr-node1-extractor`;
3. chạy regression/contract/W1 freeze gates trước khi gọi model;
4. launch **M1-01 Vanilla vLLM** bằng repo script;
5. smoke 3 sample W1;
6. chạy đủ 30 sample W1;
7. kiểm `schema_valid_rate >= 0.95` (với 30 mẫu cần ít nhất 29 valid).

> Đây là correctness/acceptance run của M1-03, **không phải** benchmark concurrency M1-02.

## Kaggle settings trước khi Run All

- Accelerator: **GPU T4 x2** (canonical target của notebook này).
- Internet: **ON** để clone repo và tải model.
- Secret `GH_PAT`: GitHub PAT có quyền đọc repo private.
- Optional secret `HF_TOKEN`: chỉ dùng cho Hugging Face download/rate-limit, không in ra log.

In [4]:
!nvidia-smi
!python --version

Sun Sep 27 07:49:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import importlib.metadata as metadata
import torch

print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "CUDA unavailable: enable Kaggle GPU first."

print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))

for package in ["vllm", "jsonschema", "pydantic"]:
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "NOT INSTALLED")

torch: 2.13.0+cu130
torch CUDA: 13.0
CUDA available: True
GPU count: 2
0 Tesla T4 (7, 5)
1 Tesla T4 (7, 5)
vllm 0.29.0
jsonschema 4.26.0
pydantic 2.12.3


## Clone đúng M1-03 branch

Cell này dùng đúng pattern `GH_PAT` như notebook M1-02 bạn đã chạy. Nó chỉ xoá clone cũ trong `/kaggle/working/do-an-ptit-d22` nếu bạn rerun notebook.

In [7]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import os
import shutil
import subprocess

repo = Path("/kaggle/working/do-an-ptit-d22")
branch = "feat/m1-03-mc-ocr-node1-extractor"

if repo.exists():
    print("Removing old Kaggle working clone:", repo)
    shutil.rmtree(repo)

user_secrets = UserSecretsClient()
token = user_secrets.get_secret("GH_PAT")

askpass = Path("/kaggle/working/git-askpass.sh")
askpass.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8",
)
askpass.chmod(0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

subprocess.run(
    [
        "git", "clone",
        "--branch", branch,
        "--single-branch",
        "https://github.com/theflyingkhui04/do-an-ptit-d22.git",
        str(repo),
    ],
    env=env,
    check=True,
)

askpass.unlink(missing_ok=True)
print("Clone OK:", repo)

Cloning into '/kaggle/working/do-an-ptit-d22'...


Clone OK: /kaggle/working/do-an-ptit-d22


In [8]:
!git -C /kaggle/working/do-an-ptit-d22 branch --show-current
!git -C /kaggle/working/do-an-ptit-d22 rev-parse HEAD
!git -C /kaggle/working/do-an-ptit-d22 log -5 --oneline
!git -C /kaggle/working/do-an-ptit-d22 status --short

# Thin runner phải import reusable Node1Extractor.
!grep -n "Node1Extractor" /kaggle/working/do-an-ptit-d22/scripts/run_node1_extractor.py

feat/m1-03-mc-ocr-node1-extractor
f8ff030ab7ee0c00c8666dd1b125ffac9f8e2f5e
f8ff030 (HEAD -> feat/m1-03-mc-ocr-node1-extractor, origin/feat/m1-03-mc-ocr-node1-extractor) refactor(m1-03): use reusable node1 runner
4b27ffd feat(m1-03): add reusable node1 runtime validation
183d6aa refactor(pipeline): extract reusable node1 core
57badbe fix(w1): make frozen package hashing cross-platform
5903e44 Merge remote-tracking branch 'origin/main' into feat/m1-03-mc-ocr-node1-extractor
4:``src.pipeline.extractor.Node1Extractor``. Reusable parsing, canonicalization,
42:    Node1Extractor,
230:    Persistence remains outside Node1Extractor validity. The extractor validates
278:            "M1-03 Node 1 run through reusable Node1Extractor",
485:    extractor = Node1Extractor(


## Offline gates trước khi gọi model

Nếu cell này fail thì **không launch vLLM**. Sửa code/branch trước rồi mới chạy acceptance.

In [9]:
import subprocess

repo = "/kaggle/working/do-an-ptit-d22"

commands = [
    ["python", "-m", "unittest", "discover", "-s", "tests", "-v"],
    ["python", "scripts/check_invoice_contract.py"],
    ["python", "scripts/check_w1_dataset.py"],
    ["python", "-m", "py_compile", "scripts/run_node1_extractor.py"],
]

for cmd in commands:
    print("\n$", " ".join(cmd))
    subprocess.run(cmd, cwd=repo, check=True)

print("\nOFFLINE GATES: PASS")


$ python -m unittest discover -s tests -v


test_business_values_and_source_are_not_changed (test_extractor.CanonicalizationTests.test_business_values_and_source_are_not_changed) ... ok
test_composes_issued_at (test_extractor.CanonicalizationTests.test_composes_issued_at) ... ok
test_defaults_source_label_and_removes_unsupported_evidence (test_extractor.CanonicalizationTests.test_defaults_source_label_and_removes_unsupported_evidence) ... ok
test_replaces_null_money_containers (test_extractor.CanonicalizationTests.test_replaces_null_money_containers) ... ok
test_invalid_request_row_is_run_level_dataset_error (test_extractor.Node1ExtractorRuntimeTests.test_invalid_request_row_is_run_level_dataset_error) ... ok
test_preflight_classifies_unreachable_endpoint (test_extractor.Node1ExtractorRuntimeTests.test_preflight_classifies_unreachable_endpoint) ... ok
test_preflight_passes_when_model_exists (test_extractor.Node1ExtractorRuntimeTests.test_preflight_passes_when_model_exists) ... ok
test_preflight_rejects_missing_model (test_extrac


$ python scripts/check_invoice_contract.py
{
  "task": "M1-03",
  "step": "invoice-contract-check",
  "status": "PASS",
  "json_schema": "src/pipeline/schemas/invoice_extraction.schema.json",
  "pydantic_model": "src.pipeline.invoice_models.InvoiceExtraction",
  "sqlite_user_version": 1,
  "example": "src/pipeline/examples/node1_extractor_output_m0_04_001.json"
}

$ python scripts/check_w1_dataset.py
{
  "task": "M1-03",
  "step": "check-w1-workload",
  "status": "PASS",
  "workload_id": "w1_node1_invoice_extraction",
  "dataset_version": "w1.m1_03.v1",
  "sample_count": 30,
  "request_count": 30,
  "package_sha256": "e7cba647035f8834920d523ce8f6a70ef68896069ee71d3a862fac2fc42eff66",
  "file_hashes": {
    "samples_jsonl": "56d33223f3d19b2f573cc7acf71e3b2288e36de05815f6f7073fda90ca3a4799",
    "provenance_jsonl": "37b8a7a2f6acf9202997ecc975b7663ee07d1bae9df29320894d649b14a8ac80",
    "manifest_csv": "8ee660c2e82bee6a90e685a64e9b9e53ca112d866d471979131aa1ec4b9a5bfd",
    "node1_inputs_

## Environment contract

Notebook M1-02 của bạn đã chạy thành công với Tesla T4, vLLM 0.29.0 và CUDA 13.0.
Cell dưới đây giữ gate rõ ràng thay vì âm thầm chạy trên hardware khác.

Nếu Kaggle cấp P100 thay vì T4, **không dùng run đó làm canonical M1 evidence**.

In [10]:
import importlib.metadata as metadata
import torch

gpu_name = torch.cuda.get_device_name(0)
vllm_version = metadata.version("vllm")
torch_version = torch.__version__
torch_cuda = torch.version.cuda

print("gpu:", gpu_name)
print("vllm:", vllm_version)
print("torch:", torch_version)
print("torch CUDA:", torch_cuda)

assert "T4" in gpu_name, f"Expected T4 for canonical run, got {gpu_name!r}"
assert vllm_version == "0.29.0", (
    f"Expected vLLM 0.29.0, got {vllm_version}. "
    "Do not silently change the experimental environment."
)

print("M1-03 TARGET ENVIRONMENT GATE: PASS")

gpu: Tesla T4
vllm: 0.29.0
torch: 2.13.0+cu130
torch CUDA: 13.0
M1-03 TARGET ENVIRONMENT GATE: PASS


## Launch Vanilla vLLM

Dùng **repo script** `scripts/launch_lane1_vanilla.sh` thay vì tự viết server args trong notebook.
Server chỉ thấy GPU 0 để tránh thay đổi execution setup ngoài ý muốn.

In [15]:
from pathlib import Path

path = Path(
    "/kaggle/working/do-an-ptit-d22/"
    "scripts/launch_lane1_vanilla.sh"
)

text = path.read_text(encoding="utf-8")

old = "--disable-log-requests"
new = "--no-enable-log-requests"

assert old in text, f"{old!r} not found in {path}"

path.write_text(
    text.replace(old, new),
    encoding="utf-8",
)

print(path.read_text(encoding="utf-8"))

#!/usr/bin/env bash
set -e

echo "  KHỞI CHẠY vLLM LÀN 1: VANILLA BASELINE (Qwen2.5-7B-Instruct AWQ)"

# 1. Cấu hình biến môi trường tối ưu bộ nhớ
export HF_HUB_ENABLE_HF_TRANSFER=0

# 2. Xác định lệnh vLLM (hỗ trợ cả CLI và Python module nếu chưa cấu hình PATH)
if command -v vllm &> /dev/null; then
  VLLM_CMD="vllm serve"
else
  VLLM_CMD="python3 -m vllm.entrypoints.openai.api_server"
fi

# 3. Khởi chạy máy chủ vLLM OpenAI-compatible
$VLLM_CMD Qwen/Qwen2.5-7B-Instruct-AWQ \
  --host 0.0.0.0 \
  --port 8001 \
  --served-model-name qwen2.5-7b-vanilla \
  --quantization awq \
  --dtype half \
  --gpu-memory-utilization 0.85 \
  --max-model-len 4096 \
  --max-num-seqs 32 \
  --no-enable-log-requests


In [16]:
!vllm serve --help | grep -A2 -B2 "log-requests"

In [17]:
!vllm serve --help 2>&1 | grep -A2 -B2 -E "enable-log-requests|no-enable-log-requests"

In [18]:
import os
import subprocess
from pathlib import Path

repo = Path("/kaggle/working/do-an-ptit-d22")
log_path = Path("/kaggle/working/m1_03_vllm_server.log")

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"

log_file = open(log_path, "w")

server_process = subprocess.Popen(
    ["bash", "scripts/launch_lane1_vanilla.sh"],
    cwd=repo,
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("server PID:", server_process.pid)

server PID: 357


## Wait for health

Không đo sample nào trước khi `/health` ready.

In [26]:
!tail -n 100 /kaggle/working/m1_03_vllm_server.log

  KHỞI CHẠY vLLM LÀN 1: VANILLA BASELINE (Qwen2.5-7B-Instruct AWQ)
(APIServer pid=358) INFO 09-27 08:03:15 [api_utils.py:347] 
(APIServer pid=358) INFO 09-27 08:03:15 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=358) INFO 09-27 08:03:15 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.29.0
(APIServer pid=358) INFO 09-27 08:03:15 [api_utils.py:347]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-7B-Instruct-AWQ
(APIServer pid=358) INFO 09-27 08:03:15 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=358) INFO 09-27 08:03:15 [api_utils.py:347] 
(APIServer pid=358) INFO 09-27 08:03:15 [api_utils.py:286] non-default args: {'model_tag': 'Qwen/Qwen2.5-7B-Instruct-AWQ', 'host': '0.0.0.0', 'port': 8001, 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ', 'dtype': 'half', 'max_model_len': 4096, 'quantization': 'awq', 'served_model_name': ['qwen2.5-7b-vanilla'], 'gpu_memory_utilization': 0.85, 'max_num_seqs': 32}
(APIServer pid=358) Warning: You are sending unauthenticat

In [27]:
import time
import urllib.request

health_url = "http://127.0.0.1:8001/health"
deadline = time.time() + 600

last_error = None
while time.time() < deadline:
    try:
        with urllib.request.urlopen(health_url, timeout=5) as response:
            status = response.status
        if status == 200:
            print("vLLM HEALTH PASS:", status)
            break
    except Exception as exc:
        last_error = exc
        time.sleep(5)
else:
    raise RuntimeError(f"vLLM did not become healthy: {last_error}")

vLLM HEALTH PASS: 200


In [28]:
!curl -sS http://127.0.0.1:8001/v1/models | python -m json.tool

{
    "object": "list",
    "data": [
        {
            "id": "qwen2.5-7b-vanilla",
            "object": "model",
            "created": 1790496353,
            "owned_by": "vllm",
            "root": "Qwen/Qwen2.5-7B-Instruct-AWQ",
            "parent": null,
            "max_model_len": 4096,
            "permission": [
                {
                    "id": "modelperm-a2af610f49617546",
                    "object": "model_permission",
                    "created": 1790496353,
                    "allow_create_engine": false,
                    "allow_sampling": true,
                    "allow_logprobs": true,
                    "allow_search_indices": false,
                    "allow_view": true,
                    "allow_fine_tuning": false,
                    "organization": "*",
                    "group": null,
                    "is_blocking": false
                }
            ]
        }
    ]
}


## Smoke — 3 W1 samples

Đây chỉ là functional smoke. Không dùng 3 mẫu để claim `>=95%`.

In [29]:
import subprocess
from pathlib import Path

repo = Path("/kaggle/working/do-an-ptit-d22")

cmd = [
    "python", "scripts/run_node1_extractor.py",
    "--requests", "experiments/configs/w1/node1_chat_requests.jsonl",
    "--package-manifest", "experiments/configs/w1/package_manifest.json",
    "--base-url", "http://127.0.0.1:8001/v1",
    "--model", "qwen2.5-7b-vanilla",
    "--run-id", "m1_03_w1_vanilla_smoke",
    "--lane", "vanilla",
    "--limit", "3",
    "--no-db",
    "--output-dir", "experiments/runs/m1_03_w1_vanilla_smoke",
    "--report", "experiments/results/m1_03_node1_vllm_smoke_summary.json",
]

print("$", " ".join(cmd))
smoke = subprocess.run(cmd, cwd=repo)
print("smoke return code:", smoke.returncode)

$ python scripts/run_node1_extractor.py --requests experiments/configs/w1/node1_chat_requests.jsonl --package-manifest experiments/configs/w1/package_manifest.json --base-url http://127.0.0.1:8001/v1 --model qwen2.5-7b-vanilla --run-id m1_03_w1_vanilla_smoke --lane vanilla --limit 3 --no-db --output-dir experiments/runs/m1_03_w1_vanilla_smoke --report experiments/results/m1_03_node1_vllm_smoke_summary.json
[1/3] OK m0_04_001 (20017 ms)
[2/3] OK m0_04_002 (17613 ms)
[3/3] OK m0_04_003 (18243 ms)
{
  "task": "M1-03",
  "step": "run-node1-extractor",
  "backend": "openai-compatible",
  "lane": "vanilla",
  "base_url": "http://127.0.0.1:8001/v1",
  "model": "qwen2.5-7b-vanilla",
  "run_id": "m1_03_w1_vanilla_smoke",
  "schema_version": "invoice_extraction.v1",
  "prompt_version": "node1_extractor.v1",
  "workload_id": "w1_node1_invoice_extraction",
  "dataset_version": "w1.m1_03.v1",
  "w1_package_sha256": "e7cba647035f8834920d523ce8f6a70ef68896069ee71d3a862fac2fc42eff66",
  "artifact_hash

In [30]:
import json
from pathlib import Path

repo = Path("/kaggle/working/do-an-ptit-d22")
path = repo / "experiments/results/m1_03_node1_vllm_smoke_summary.json"

summary = json.loads(path.read_text(encoding="utf-8"))
print(json.dumps(summary, ensure_ascii=False, indent=2))

assert summary["request_count"] == 3
assert summary["w1_package_sha256"] == (
    "e7cba647035f8834920d523ce8f6a70ef68896069ee71d3a862fac2fc42eff66"
)
print("SMOKE ARTIFACT IDENTITY: PASS")

{
  "task": "M1-03",
  "step": "run-node1-extractor",
  "backend": "openai-compatible",
  "lane": "vanilla",
  "base_url": "http://127.0.0.1:8001/v1",
  "model": "qwen2.5-7b-vanilla",
  "run_id": "m1_03_w1_vanilla_smoke",
  "schema_version": "invoice_extraction.v1",
  "prompt_version": "node1_extractor.v1",
  "workload_id": "w1_node1_invoice_extraction",
  "dataset_version": "w1.m1_03.v1",
  "w1_package_sha256": "e7cba647035f8834920d523ce8f6a70ef68896069ee71d3a862fac2fc42eff66",
  "artifact_hashes": {
    "samples_jsonl": "56d33223f3d19b2f573cc7acf71e3b2288e36de05815f6f7073fda90ca3a4799",
    "provenance_jsonl": "37b8a7a2f6acf9202997ecc975b7663ee07d1bae9df29320894d649b14a8ac80",
    "manifest_csv": "8ee660c2e82bee6a90e685a64e9b9e53ca112d866d471979131aa1ec4b9a5bfd",
    "node1_inputs_jsonl": "1cd69a43177375e2a3dab5ba4b4b27ca233f1288a49211c803086756ef7219ce",
    "node1_chat_requests_jsonl": "0de2387c4e6a8c39523bf869788a55ee7487c6a32318a949ce22e1f8f7b17360",
    "system_prompt": "3452c77

# Final M1-03 acceptance — full 30 W1 samples

Acceptance metric:

`schema_valid_rate = fully_validated_count / request_count`

Với 30 mẫu:
- 29/30 = 96.67% → PASS
- 28/30 = 93.33% → FAIL

Không xoá failed sample khỏi denominator.

In [31]:
import subprocess
from pathlib import Path

repo = Path("/kaggle/working/do-an-ptit-d22")

cmd = [
    "python", "scripts/run_node1_extractor.py",
    "--requests", "experiments/configs/w1/node1_chat_requests.jsonl",
    "--package-manifest", "experiments/configs/w1/package_manifest.json",
    "--base-url", "http://127.0.0.1:8001/v1",
    "--model", "qwen2.5-7b-vanilla",
    "--run-id", "m1_03_w1_vanilla",
    "--lane", "vanilla",
    "--no-db",
    "--output-dir", "experiments/runs/m1_03_w1_vanilla",
    "--report", "experiments/results/m1_03_node1_vllm_summary.json",
    "--min-schema-valid-rate", "0.95",
]

print("$", " ".join(cmd))
final_run = subprocess.run(cmd, cwd=repo)

# Do not immediately raise: inspect evidence even when acceptance fails.
print("final return code:", final_run.returncode)
print("0 = run completed and gate passed")
print("2 = schema_valid_rate below threshold")

$ python scripts/run_node1_extractor.py --requests experiments/configs/w1/node1_chat_requests.jsonl --package-manifest experiments/configs/w1/package_manifest.json --base-url http://127.0.0.1:8001/v1 --model qwen2.5-7b-vanilla --run-id m1_03_w1_vanilla --lane vanilla --no-db --output-dir experiments/runs/m1_03_w1_vanilla --report experiments/results/m1_03_node1_vllm_summary.json --min-schema-valid-rate 0.95
[1/30] OK m0_04_001 (18025 ms)
[2/30] OK m0_04_002 (19950 ms)
[3/30] OK m0_04_003 (19066 ms)
[4/30] OK m0_04_004 (18337 ms)
[5/30] OK m0_04_005 (18265 ms)
[6/30] OK m0_04_006 (18635 ms)
[7/30] OK m0_04_007 (19402 ms)
[8/30] OK m0_04_008 (17003 ms)
[9/30] OK m0_04_009 (18746 ms)
[10/30] OK m0_04_010 (19144 ms)
[11/30] OK m0_04_011 (19187 ms)
[12/30] OK m0_04_012 (18871 ms)
[13/30] OK m0_04_013 (17000 ms)
[14/30] OK m0_04_014 (19062 ms)
[15/30] OK m0_04_015 (19200 ms)
[16/30] OK m0_04_016 (18979 ms)
[17/30] OK m0_04_017 (19047 ms)
[18/30] OK m0_04_018 (20554 ms)
[19/30] OK m0_04_019 (

In [32]:
import json
from pathlib import Path

repo = Path("/kaggle/working/do-an-ptit-d22")
summary_path = repo / "experiments/results/m1_03_node1_vllm_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))

print(json.dumps(summary, ensure_ascii=False, indent=2))

attempted = summary["request_count"]
valid = summary["fully_validated_count"]
rate = summary["rates"]["schema_valid_rate"]

assert attempted == 30, f"Expected frozen W1 30 requests, got {attempted}"
assert summary["w1_package_sha256"] == (
    "e7cba647035f8834920d523ce8f6a70ef68896069ee71d3a862fac2fc42eff66"
)
assert rate == round(valid / attempted, 6)

print()
print(f"VALID: {valid}/{attempted} = {rate:.2%}")
print("ACCEPTANCE:", "PASS" if rate >= 0.95 else "FAIL")
print("failure_counts:", summary["failure_counts"])

{
  "task": "M1-03",
  "step": "run-node1-extractor",
  "backend": "openai-compatible",
  "lane": "vanilla",
  "base_url": "http://127.0.0.1:8001/v1",
  "model": "qwen2.5-7b-vanilla",
  "run_id": "m1_03_w1_vanilla",
  "schema_version": "invoice_extraction.v1",
  "prompt_version": "node1_extractor.v1",
  "workload_id": "w1_node1_invoice_extraction",
  "dataset_version": "w1.m1_03.v1",
  "w1_package_sha256": "e7cba647035f8834920d523ce8f6a70ef68896069ee71d3a862fac2fc42eff66",
  "artifact_hashes": {
    "samples_jsonl": "56d33223f3d19b2f573cc7acf71e3b2288e36de05815f6f7073fda90ca3a4799",
    "provenance_jsonl": "37b8a7a2f6acf9202997ecc975b7663ee07d1bae9df29320894d649b14a8ac80",
    "manifest_csv": "8ee660c2e82bee6a90e685a64e9b9e53ca112d866d471979131aa1ec4b9a5bfd",
    "node1_inputs_jsonl": "1cd69a43177375e2a3dab5ba4b4b27ca233f1288a49211c803086756ef7219ce",
    "node1_chat_requests_jsonl": "0de2387c4e6a8c39523bf869788a55ee7487c6a32318a949ce22e1f8f7b17360",
    "system_prompt": "3452c77da3a64

## Artifact sanity check

Mỗi sample phải join được theo `sample_id`; raw response nếu có được giữ riêng khỏi canonical validated output.

In [33]:
import json
from pathlib import Path

repo = Path("/kaggle/working/do-an-ptit-d22")
run_dir = repo / "experiments/runs/m1_03_w1_vanilla"

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

raw = read_jsonl(run_dir / "raw_responses.jsonl")
valid = read_jsonl(run_dir / "validated_outputs.jsonl")
fail = read_jsonl(run_dir / "failures.jsonl")

print("raw responses:", len(raw))
print("validated outputs:", len(valid))
print("failures:", len(fail))

print("validated sample IDs:", [row["sample_id"] for row in valid[:5]])
print("failure sample IDs:", [row["sample_id"] for row in fail[:5]])

assert len(valid) + sum(
    1 for row in fail if row.get("error_code") != "PERSISTENCE_ERROR"
) == 30

print("ARTIFACT DENOMINATOR CHECK: PASS")

raw responses: 30
validated outputs: 30
failures: 0
validated sample IDs: ['m0_04_001', 'm0_04_002', 'm0_04_003', 'm0_04_004', 'm0_04_005']
failure sample IDs: []
ARTIFACT DENOMINATOR CHECK: PASS


## Pack evidence để tải xuống / lưu làm Kaggle output

In [34]:
!cd /kaggle/working/do-an-ptit-d22 && \
zip -r /kaggle/working/m1_03_node1_vllm_evidence.zip \
  experiments/runs/m1_03_w1_vanilla \
  experiments/results/m1_03_node1_vllm_summary.json

!ls -lh /kaggle/working/m1_03_node1_vllm_evidence.zip

  adding: experiments/runs/m1_03_w1_vanilla/ (stored 0%)
  adding: experiments/runs/m1_03_w1_vanilla/summary.json (deflated 54%)
  adding: experiments/runs/m1_03_w1_vanilla/failures.jsonl (stored 0%)
  adding: experiments/runs/m1_03_w1_vanilla/raw_responses.jsonl (deflated 94%)
  adding: experiments/runs/m1_03_w1_vanilla/validated_outputs.jsonl (deflated 94%)
  adding: experiments/results/m1_03_node1_vllm_summary.json (deflated 54%)
-rw-r--r-- 1 root root 15K Sep 27 08:19 /kaggle/working/m1_03_node1_vllm_evidence.zip


## Stop server khi xong

In [ ]:
server_process.terminate()
try:
    server_process.wait(timeout=30)
except Exception:
    server_process.kill()

log_file.close()
print("vLLM server stopped")